# AI Explanations: Agent Platform を使用した説明可能な画像モデルのデプロイ

## 概要

このラボでは、画像データで分類モデルをトレーニングし、それを Agent Platform にデプロイして説明（特徴量属性 / Feature Attributions）付きの予測を提供する方法を示します。このラボでは次のことを行います:
* データセットの探索
* Agent Platform によるカスタム画像分類モデルの構築とトレーニング
* エンドポイントへのモデルのデプロイ
* 説明付き予測の提供
* 統合勾配（Integrated Gradients）による特徴量属性の可視化

## セットアップ

### ライブラリのインポート

このチュートリアル用のライブラリをインポートします。

In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
import base64
import random
from datetime import datetime
import io
from io import BytesIO

import matplotlib.image as mpimg
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from google.cloud import aiplatform
from matplotlib import pyplot as plt

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT  # defaults to PROJECT
REGION = "us-central1"  # Replace with your REGION

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
BUCKET = PROJECT
REGION = "us-central1"
GCS_PATTERN = "gs://asl-public-data/data/flowers-xai/*.tfrec"
DATA_PATH = f"gs://{BUCKET}/flowers/data"
OUTDIR = f"gs://{BUCKET}/flowers/model_{TIMESTAMP}"

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION
os.environ["DATA_PATH"] = DATA_PATH
os.environ["OUTDIR"] = OUTDIR
os.environ["TIMESTAMP"] = TIMESTAMP
print(f"Project: {PROJECT}")

Cloud Storage バケットがまだ存在しない場合は、次のセルを実行して作成します。

In [ ]:
%%bash
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
   echo -e "Bucket gs://${BUCKET} already exists."
    
else
   echo "Creating a new GCS bucket."
   gcloud storage buckets create --location=${REGION} gs://${BUCKET}
   echo -e "\nHere are your current buckets:"
   gcloud storage ls
fi

## データセットの探索
このチュートリアルで使用するデータセットは、[TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/overview) の [flowers データセット](https://www.tensorflow.org/datasets/catalog/tf_flowers) です。このセクションでは、ファイルをシャッフル、分割して GCS バケットにコピーする方法を説明します。

### データセットの読み込み、分割、および GCS バケットへのコピー

In [ ]:
TRAINING_DATA_PATH = DATA_PATH + "/training"
EVAL_DATA_PATH = DATA_PATH + "/validation"
VALIDATION_SPLIT = 0.2

# Split data files between training and validation
filenames = tf.io.gfile.glob(GCS_PATTERN)
random.shuffle(filenames)
split = int(len(filenames) * VALIDATION_SPLIT)
training_filenames = filenames[split:]
validation_filenames = filenames[:split]

# Copy training files to GCS
for file in training_filenames:
    !gcloud storage cp $file $TRAINING_DATA_PATH/

# Copy eval files to GCS
for file in validation_filenames:
    !gcloud storage cp $file $EVAL_DATA_PATH/

以下のコマンドを実行します。GCS バケットの gs://{BUCKET}/flowers/data/training と gs://{BUCKET}/flowers/data/validation の両方に多数の .tfrec ファイルが表示されるはずです。

In [ ]:
!gcloud storage ls -l $TRAINING_DATA_PATH

In [ ]:
!gcloud storage ls -l $EVAL_DATA_PATH

### 取り込み関数の作成とサンプルの可視化
画像と対応するラベルをプロットするためのヘルパー関数を定義して実行します。

In [ ]:
IMAGE_SIZE = [192, 192]
BATCH_SIZE = 32

# Do not change, maps to the labels in the data
CLASSES = [
    "daisy",
    "dandelion",
    "roses",
    "sunflowers",
    "tulips",
]


def read_tfrecord(example):
    features = {
        "image": tf.io.FixedLenFeature(
            [], tf.string
        ),  # tf.string means bytestring
        "class": tf.io.FixedLenFeature([], tf.int64),  # shape [] means scalar
        "one_hot_class": tf.io.VarLenFeature(tf.float32),
    }
    example = tf.io.parse_single_example(example, features)
    image = tf.image.decode_jpeg(example["image"], channels=3)
    image = (
        tf.cast(image, tf.float32) / 255.0
    )  # convert image to floats in [0, 1] range
    image = tf.reshape(image, [*IMAGE_SIZE, 3])
    one_hot_class = tf.sparse.to_dense(example["one_hot_class"])
    one_hot_class = tf.reshape(one_hot_class, [5])
    return image, one_hot_class


# Load tfrecords into tf.data.Dataset
def load_dataset(gcs_pattern):
    filenames = tf.io.gfile.glob(gcs_pattern + "/*")
    ds = tf.data.TFRecordDataset(filenames).map(read_tfrecord)
    return ds


# Converts N examples in dataset to numpy arrays
def dataset_to_numpy(dataset, N):
    numpy_images = []
    numpy_labels = []

    for images, labels in dataset.take(N):
        numpy_images.append(images.numpy())
        numpy_labels.append(labels.numpy())

    return numpy_images, numpy_labels


def display_one_image(image, title, subplot):
    plt.subplot(subplot)
    plt.axis("off")
    plt.imshow(image)
    plt.title(title, fontsize=16)
    return subplot + 1


def display_9_images_from_dataset(dataset):
    subplot = 331
    plt.figure(figsize=(13, 13))
    images, labels = dataset_to_numpy(dataset, 9)
    for i, image in enumerate(images):
        title = CLASSES[np.argmax(labels[i], axis=-1)]
        subplot = display_one_image(image, title, subplot)
        if i >= 8:
            break

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.1, hspace=0.1)
    plt.show()


# Display 9 examples from the dataset
ds = load_dataset(gcs_pattern=TRAINING_DATA_PATH)
display_9_images_from_dataset(ds)

## トレーニングパイプラインの構築
このセクションでは、Agent Platform Custom Training 上で画像分類モデルをトレーニングするための Keras アプリケーションを構築します。

トレーニングアプリケーション用のディレクトリと `__init__.py` ファイルを作成します（これは Python アプリケーションに必要ですが、空でも構いません）。

In [ ]:
%%bash 
mkdir -p flowers/trainer
touch flowers/trainer/__init__.py

### train.py でのトレーニングアプリケーションの作成

このコードにはトレーニングロジックが含まれています。ここでは、GCS からデータを取り込み、特徴量抽出器として [mobileNet](https://tfhub.dev/google/imagenet/mobilenet_v2_100_192/feature_vector/5) を使用して画像分類モデルをトレーニングし、その出力特徴量ベクトルを 5 ユニットとソフトマックス活性化関数を持つ tf.keras.dense レイヤー（5 つのラベル候補があるため）に渡すアプリケーションを構築します。また、コマンドライン経由で `train_and_evaluate` に引数を渡すことができる `fire` ライブラリを使用します。

In [ ]:
%%writefile flowers/trainer/train.py
import datetime
import fire
import os
import tensorflow as tf
import tensorflow_hub as hub

IMAGE_SIZE = [192, 192]


def read_tfrecord(example):

    features = {
        "image": tf.io.FixedLenFeature(
            [], tf.string
        ),  # tf.string means bytestring
        "class": tf.io.FixedLenFeature([], tf.int64),  # shape [] means scalar
        "one_hot_class": tf.io.VarLenFeature(tf.float32),
    }
    example = tf.io.parse_single_example(example, features)
    image = tf.image.decode_jpeg(example["image"], channels=3)
    image = (
        tf.cast(image, tf.float32) / 255.0
    )  # convert image to floats in [0, 1] range
    image = tf.reshape(
        image, [*IMAGE_SIZE, 3]
    )
    one_hot_class = tf.sparse.to_dense(example["one_hot_class"])
    one_hot_class = tf.reshape(one_hot_class, [5])
    return image, one_hot_class


def load_dataset(gcs_pattern, batch_size=32, training=True):
    filenames = tf.io.gfile.glob(gcs_pattern)
    ds = tf.data.TFRecordDataset(filenames).map(
        read_tfrecord).batch(batch_size)
    if training:
        return ds.repeat()
    else:
        return ds


def build_model():
    # MobileNet model for feature extraction
    mobilenet_v2 = 'https://tfhub.dev/google/imagenet/'\
        'mobilenet_v2_100_192/feature_vector/5'
    feature_extractor_layer = hub.KerasLayer(
        mobilenet_v2,
        input_shape=[*IMAGE_SIZE, 3],
        trainable=False
    )

    # Instantiate model
    model = tf.keras.Sequential([
        feature_extractor_layer,
        tf.keras.layers.Dense(5, activation="softmax")
    ])

    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])

    return model


def train_and_evaluate(train_data_path,
                       eval_data_path,
                       output_dir,
                       batch_size,
                       num_epochs,
                       train_examples):

    model = build_model()
    train_ds = load_dataset(gcs_pattern=train_data_path,
                            batch_size=batch_size)
    eval_ds = load_dataset(gcs_pattern=eval_data_path,
                           training=False)
    num_batches = batch_size * num_epochs
    steps_per_epoch = train_examples // num_batches

    history = model.fit(
        train_ds,
        validation_data=eval_ds,
        epochs=num_epochs,
        steps_per_epoch=steps_per_epoch,
        verbose=2,  # 0=silent, 1=progress bar, 2=one line per epoch
    )

    model.save(output_dir)

    print("Exported trained model to {}".format(output_dir))

if __name__ == "__main__":
    fire.Fire(train_and_evaluate)


### ローカルでのトレーニングアプリケーションのテスト
コードが想定どおりに実行されることを確認するために、ローカルで（わずかなトレーニングステップ数で）トレーニングアプリケーションをテストすることをお勧めします。

In [ ]:
%%bash
OUTDIR_LOCAL=local_test_training
rm -rf ${OUTDIR_LOCAL}
export PYTHONPATH=${PYTHONPATH}:${PWD}/flowers
python3 -m trainer.train \
    --train_data_path=gs://${BUCKET}/flowers/data/training/*.tfrec \
    --eval_data_path=gs://${BUCKET}/flowers/data/validation/*.tfrec \
    --output_dir=${OUTDIR_LOCAL} \
    --batch_size=1 \
    --num_epochs=1 \
    --train_examples=10 

## コードをソースディストリビューションとしてパッケージ化
モデルのトレーニングコードを検証できたので、Agent Platform にカスタムトレーニングジョブを送信するためにコードをソースディストリビューションとしてパッケージ化する必要があります。

In [ ]:
%%writefile flowers/setup.py
from setuptools import find_packages
from setuptools import setup

setup(
    name='flowers_trainer',
    version='0.1',
    packages=find_packages(),
    include_package_data=True,
    install_requires=['fire==0.4.0', 'tensorflow-hub==0.12.0'],
    description='Flowers image classifier training application.'
)

In [ ]:
%%bash
cd flowers
python ./setup.py sdist --formats=gztar
cd ..

パッケージを GCS に保存

In [ ]:
%%bash
gcloud storage cp flowers/dist/flowers_trainer-0.1.tar.gz gs://${BUCKET}/flowers/

Cloud に送信するには、[`gcloud custom-jobs create`](https://cloud.google.com/sdk/gcloud/reference/ai/custom-jobs/create) を使用し、Agent Platform Training Service の追加パラメータをいくつか指定するだけです:
- display-name: Cloud ジョブの一意の識別子。一意性を確保するために通常はシステム時刻を追加します
- region: トレーニングを行う Cloud リージョン。サポートされている Agent Platform Training Service リージョンについては[こちら](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/general/locations)を参照してください

以前に `gcloud ai custom-jobs create` をコマンド呼び出しで直接 `worker pool spec` とパススルー Python 引数を指定して実行したのを見たことがあるかもしれませんが、ここでは YAML ファイルを使用します。これによりハイパーパラメータ調整への移行が容易になります。

`args:` 引数を通じて、`task.py` ファイル用のパススルー引数を追加します。

In [ ]:
%%bash
JOB_NAME=flowers_${TIMESTAMP}

PYTHON_PACKAGE_URI=gs://${BUCKET}/flowers/flowers_trainer-0.1.tar.gz
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-11.py310:latest"
PYTHON_MODULE=trainer.train

echo > ./config.yaml \
"workerPoolSpecs:
  machineSpec:
    machineType: n1-standard-8
  replicaCount: 1
  pythonPackageSpec:
    executorImageUri: $PYTHON_PACKAGE_EXECUTOR_IMAGE_URI
    packageUris: $PYTHON_PACKAGE_URI
    pythonModule: $PYTHON_MODULE
    args:
    - --train_data_path=gs://${BUCKET}/flowers/data/training/*.tfrec
    - --eval_data_path=gs://${BUCKET}/flowers/data/validation/*.tfrec
    - --output_dir=$OUTDIR
    - --num_epochs=15
    - --train_examples=15000
    - --batch_size=32
    "

gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --config=config.yaml

**注** モデルのトレーニングには約 5 分かかります。先へ進む前にトレーニングが完了するまでお待ちください。

### 画像データ用のサービング関数

予測サービスに画像を渡すには、圧縮された（例: JPEG）画像バイトを base64 にエンコードします。これにより、ネットワーク経由でバイナリデータを送信する際の改ざんからコンテンツを保護できます。このデプロイされたモデルは入力データが生の（非圧縮）バイトであることを想定しているため、base64 エンコードされたデータがデプロイされたモデルへの入力として渡される前に生のバイトに逆変換されるようにする必要があります。

これを解決するために、サービング関数（`serving_fn`）を定義し、前処理ステップとしてモデルにアタッチします。サービング関数が（CPU 上の上流ではなく）基盤となるモデルと融合するように、`@tf.function` デコレータを追加します。

予測または説明のリクエストを送信すると、リクエストのコンテンツは TensorFlow 文字列（`tf.string`）に base64 デコードされ、サービング関数（`serving_fn`）に渡されます。サービング関数は、モデルの入力要件に合わせて `tf.string` を生の（非圧縮）numpy バイトに前処理します（`preprocess_fn`）:
- `io.decode_jpeg`: JPG 画像を解凍し、3 チャネル（RGB）の TensorFlow テンソルとして返します。
- `image.convert_image_dtype`: 整数のピクセル値を float32 に変更します。
- `image.resize`: モデルの入力形状に合わせて画像のサイズを変更します。
- `resized / 255.0`: ピクセルデータを 0 と 1 の間にリスケール（正規化）します。

この時点で、データをモデル（`m_call`）に渡すことができます。

#### XAI シグネチャ

サービング関数を基盤となるモデルとともに再保存（`tf.saved_model.save`）する際、サービング関数の入力レイヤーをシグネチャ `serving_default` として指定します。

XAI 画像モデルの場合、サービング関数から 2 つの追加のシグネチャを保存する必要があります:

- `xai_preprocess`: サービング関数内の前処理関数。
- `xai_model`: モデルを呼び出すための具象関数（concrete function）。

モデルをメモリに読み込みます。**注** モデルのトレーニングが完了していない場合、このディレクトリは存在しません。先へ進む前にトレーニングが完了するまでお待ちください。

In [ ]:
local_model = tf.keras.models.load_model(OUTDIR)

In [ ]:
local_model.summary()

In [ ]:
CONCRETE_INPUT = "numpy_inputs"


def _preprocess(bytes_input):
    decoded = tf.io.decode_jpeg(bytes_input, channels=3)
    decoded = tf.image.convert_image_dtype(decoded, tf.float32)
    resized = #TODO: デコードされた画像をリサイズします
    return resized


@tf.function(input_signature=[tf.TensorSpec([None], tf.string)])
def preprocess_fn(bytes_inputs):
    decoded_images = tf.map_fn(
        _preprocess, bytes_inputs, dtype=tf.float32, back_prop=False
    )
    return {
        CONCRETE_INPUT: decoded_images
    }  # ユーザーはキーがモデルの入力と一致していることを確認する必要があります


@tf.function(input_signature=[tf.TensorSpec([None], tf.string)])
def serving_fn(bytes_inputs):
    images = preprocess_fn(bytes_inputs)
    prob = m_call(**images)
    return prob


# モデル自体にデータを送信し、出力確率を返す関数
m_call = tf.function(local_model.call).get_concrete_function(
    [
        tf.TensorSpec(
            shape=[None, 192, 192, 3], dtype=tf.float32, name=CONCRETE_INPUT
        )
    ]
)

tf.saved_model.save(
    local_model,
    OUTDIR,
    signatures={
        "serving_default": serving_fn,
        # XAI に必要
        "xai_preprocess": preprocess_fn,
        "xai_model": m_call,
    },
)

## サービング関数のシグネチャの取得

モデルをメモリに再読み込みし、各レイヤーに対応するシグネチャをクエリすることで、モデルの入力レイヤーと出力レイヤーのシグネチャを取得できます。

予測リクエストを行う際、リクエストをモデルではなくサービング関数にルーティングする必要があるため、サービング関数の入力レイヤー名を知る必要があります（これは後で予測リクエストを行う際に使用します）。

また、後述する説明用メタデータを構築するために、サービング関数の入力レイヤーと出力レイヤーの名前を知る必要があります。

In [ ]:
loaded = tf.saved_model.load(OUTDIR)

serving_input = list(
    loaded.signatures["serving_default"].structured_input_signature[1].keys()
)[0]
print("Serving function input:", serving_input)
serving_output = list(
    loaded.signatures["serving_default"].structured_outputs.keys()
)[0]
print("Serving function output:", serving_output)

input_name = local_model.input.name
print("Model input name:", input_name)
output_name = local_model.output.name
print("Model output name:", output_name)

In [ ]:
parameters = aiplatform.explain.ExplanationParameters(
    {"integrated_gradients_attribution": {"step_count": 50}}
)

## モデルのアップロード

次に、以下のパラメータを指定して `Model.upload()` メソッドを使用し、モデルを `Model` リソースにアップロードします:

- `display_name`: `Model` リソースの人間が読める名前。
- `artifact`: トレーニング済みモデルのアーティファクトの Cloud Storage の場所。
- `serving_container_image_uri`: サービングコンテナイメージ。
- `sync`: アップロードを非同期で実行するか同期で実行するか。
- `explanation_parameters`: Parameters to configure explaining for `Model`'s predictions.
- `explanation_metadata`: Metadata describing the `Model`'s input and output for explanation.

`upload()` メソッドを非同期で実行した場合は、後から `wait()` メソッドを使用して完了するまで待機（ブロック）できます。

In [ ]:
MODEL_NAME = "flower_classifier_v1"
INPUT_METADATA = {"input_tensor_name": CONCRETE_INPUT, "modality": "image"}
OUTPUT_METADATA = {"output_tensor_name": serving_output}

input_metadata = aiplatform.explain.ExplanationMetadata.InputMetadata(
    INPUT_METADATA
)
output_metadata = aiplatform.explain.ExplanationMetadata.OutputMetadata(
    OUTPUT_METADATA
)

metadata = aiplatform.explain.ExplanationMetadata(
    inputs={"image": input_metadata}, outputs={"class": output_metadata}
)

**注** これには実行に数分かかる場合があります。

In [ ]:
aiplatform.init(project=PROJECT, staging_bucket=BUCKET)
model = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=OUTDIR,
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-11:latest",
    explanation_parameters=parameters,
    explanation_metadata=metadata,
    sync=False,
)

model.wait()

## モデルのデプロイ

次に、オンライン予測用にモデルをデプロイします。モデルをデプロイするには、以下のパラメータを指定して `deploy` メソッドを呼び出します:

- `deployed_model_display_name`: デプロイされたモデルの人間が読める名前。
- `traffic_split`: エンドポイントでこのモデルに送られるトラフィックの割合（パーセント）。1 つ以上のキーと値のペアの辞書として指定します。
モデルが 1 つだけの場合は `{ "0": 100 }` として指定します（ここで "0" はアップロードされるこのモデルを指し、100 はトラフィックの 100% を意味します）。
トラフィックを分割する既存のモデルがエンドポイントにある場合は、model_id を使用して `{ "0": percent, model_id: percent, ... }` のように指定します（ここで model_id はデプロイされたエンドポイントに対する既存モデルのモデル ID です）。パーセントの合計は 100 になる必要があります。
- `machine_type`: サービングに使用するマシンのタイプ。
- `max_replica_count`: スケールアウトするコンピューティングインスタンスの最大数。このチュートリアルでは、1 つのインスタンスのみがプロビジョニングされます。

**注** これには数分かかる場合があります。

In [ ]:
endpoint = model.deploy(
    deployed_model_display_name=MODEL_NAME,
    traffic_split={"0": 100},
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=1,
)

### リクエストコンテンツの準備
生の非圧縮バイトではなく、圧縮された JPG 画像として花の画像を送信します:

- `mpimg.imsave`: 非圧縮画像を圧縮 JPEG 画像としてディスクに書き込みます。
- `tf.io.read_file`: 圧縮された JPG 画像を生のバイトとしてメモリに読み戻します。
- `base64.b64encode`: 生のバイトを base64 エンコードされた文字列にエンコードします。

In [ ]:
eval_ds = load_dataset(EVAL_DATA_PATH)
x_test, y_test = dataset_to_numpy(eval_ds, 5)

# Single image from eval dataset
test_image = x_test[0]

# Write image out as jpg
mpimg.imsave("tmp.jpg", test_image)

JPG 画像を読み込み、モデルエンドポイントに送信するために base64 でエンコードします。`endpoint.explain` を使用してエンコードされた画像をエンドポイントに送信します。その後、レスポンスをパースして予測と説明を取得できます。endpoint.explain に関する完全なドキュメントは[こちら](https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/rest)にあります。

In [ ]:
# 画像を読み込んで base64 エンコード
bytes = tf.io.read_file("tmp.jpg")
b64str = base64.b64encode(bytes.numpy()).decode("utf-8")

instances_list = [{serving_input: {"b64": b64str}}]

response = #TODO: エンドポイントから説明付きの予測を取得します
print(response)

### 統合勾配（Integrated Gradients）による特徴量属性の可視化
レスポンスを調べて予測と特徴量属性を取得します。Matplotlib を使用して可視化します。

In [ ]:
CLASSES = [
    "daisy",
    "dandelion",
    "roses",
    "sunflowers",
    "tulips",
]

# 予測をパース
for prediction in response.predictions:
    label_index = np.argmax(prediction)
    class_name = CLASSES[label_index]
    confidence_score = prediction[label_index]
    print(
        "Predicted class: "
        + class_name
        + "\n"
        + "Confidence score: "
        + str(confidence_score)
    )

    image = base64.b64decode(b64str)
    image = BytesIO(image)
    img = mpimg.imread(image, format="JPG")

# 説明（Explanation）をパース
for explanation in response.explanations:
    attributions = dict(explanation.attributions[0].feature_attributions)
    xai_label_index = explanation.attributions[0].output_index[0]
    xai_class_name = CLASSES[xai_label_index]
    xai_b64str = attributions["image"]["b64_jpeg"]
    xai_image = base64.b64decode(xai_b64str)
    xai_image = io.BytesIO(xai_image)
    xai_img = mpimg.imread(xai_image, format="JPG")

# 元画像、特徴量属性マスク、およびオーバーレイ画像をプロット
fig = plt.figure(figsize=(13, 18))
fig.add_subplot(1, 3, 1)
plt.title("Input Image")
plt.imshow(img)
fig.add_subplot(1, 3, 2)
plt.title("Feature Attribution Mask")
plt.imshow(xai_img)
fig.add_subplot(1, 3, 3)
plt.title("Overlayed Attribution Mask")
plt.imshow(img)
plt.imshow(xai_img, alpha=0.6)
plt.show()

## 次のステップ

AI Explanations の詳細については、以下のリソースを確認してください。

* [Agent Platform Explanations ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform)
* [統合勾配（Integrated Gradients）の論文](https://arxiv.org/abs/1703.01365)
* [XRAI の論文](https://arxiv.org/abs/1906.02825)